In [ ]:
import numpy as np
import pandas as pd
import serial
import time
import matplotlib.pyplot as plt
from IPython import display
from datetime import datetime
import os 
import re
import serial.tools.list_ports


plt.style.use("dark_background")

In [ ]:
# Function to find the connected devices on available serial ports
def findDevice(question="hello", answer="Ready", flush=True, timeout=3):
    """
    Find the amBIT on an available serial port by sending `question` and looking
    for `answer` in the reply. Returns the port name (str), or None if not found.
    """
    if not question.endswith("\n"):
        question += "\n"                          # firmware is line-based
    for p in serial.tools.list_ports.comports():  # only real ports, not COM1..COM256
        port = p.device
        try:
            with serial.Serial(port, baudrate=115200, timeout=timeout) as ser:
                ser.setRTS(False)                 # keep the ESP out of bootloader mode
                ser.flush()
                time.sleep(0.7)                   # let the device finish (re)booting
                if flush:
                    ser.reset_input_buffer()      # drop the boot-log / stale bytes
                ser.write(question.encode())
                time.sleep(0.5)

                reply = ""
                t0 = time.monotonic()
                while time.monotonic() - t0 < timeout:
                    line = ser.readline().decode("utf-8", errors="replace").strip()
                    if not line:
                        break
                    reply += line + "\n"
                    if answer and answer in line:
                        break

                print(f"[{port}] {p.description}: {reply.strip()!r}")
                if answer and answer in reply:
                    print(f"Found device at: {port}")
                    return port
        except (OSError, serial.SerialException):
            pass
    print("No matching device found.")
    return None

    
def ensure_path_exists(path: str) -> None:
    """
    Ensures that the given directory path exists.
    If it does not exist, this function will create it (including any intermediate directories).

    :param path: The directory path to check (and potentially create).
    """
    if not os.path.exists(path):
        print(f"Creating path: {path}")
        os.makedirs(path)
    else:
        print(f"{path} exist")

def gen_cmd_arr_line(num: int, freq: int, actinic: int) -> list:
    """
    Generate a single command-array entry of 8 parameters.
        
    :param num: Total number of measurement points (1-2000).
    :param freq: Frequency measurement points (1-200).
    :param actinic: Actinic light setting (0-255).
    :return: List of 8 integer values representing the command.
    """
    return [
        2,              # Fixed flag or identifier
        0,              # Another fixed flag or placeholder
        num // 256,     # High byte of 'num'
        num % 256,      # Low byte of 'num'
        freq // 256,    # High byte of 'freq'
        freq % 256,     # Low byte of 'freq'
        actinic,        # Actinic light parameter
        1               # Fixed flag or terminator
    ]

def calc_arr_param(cmd: list, persist: bool = False):
    """
    Reshape a flat command list into a 2D array, construct a command string,
    and calculate measurement timelines and actinic-array values.

    :param cmd: A 1D list of length 8*N (multiple commands concatenated).
    :param persist: A flag indicating whether the command persists after completion.
    :return: Tuple (cmd_str, mea_tml, mea_act)
        - cmd_str: Formatted command string (e.g., for sending over serial).
        - mea_tml: NumPy array of timestamps (scaled by 0.854).
        - mea_act: List of actinic values (one per timestamp).
    """
    # Reshape 'cmd' into an N x 8 array
    cmd_arr = np.reshape(cmd, (-1, 8))
    arr_length = cmd_arr.shape[0]

    # Create a comma-separated command string:
    # e.g., "arrun1,N,persist,<entire cmd list>,\n"
    cmd_str = f"arrun1,{arr_length},{persist},{str(cmd)[1:-1]},\n"
    cmd_str = cmd_str.replace(" ", "")  # Remove spaces for a cleaner command

    # Parse the reshaped array into usable parameters
    num_pts = cmd_arr[:, 2] * 256 + cmd_arr[:, 3]  # Number of points (high/low bytes)
    act_arr = cmd_arr[:, 6]                       # Actinic setting
    mea_feq = 1 / (cmd_arr[:, 4] * 256 + cmd_arr[:, 5])  # Measurement frequency (1 / freq)

    # Build up the measurement timeline and actinic values
    _t = 0
    mea_tml = []
    mea_act = []

    # For each command segment, create a timeline and corresponding actinic values
    for _n, _f, _a in zip(num_pts, mea_feq, act_arr):
        segment_times = (np.arange(_n) * _f) + _t
        mea_tml.extend(segment_times)       # Accumulate times
        mea_act.extend([_a] * _n)           # Repeat actinic value _n times
        _t = segment_times[-1]              # Update the offset for the next segment

    # Scale the entire timeline by 0.854
    mea_tml = np.array(mea_tml) * 0.854

    return cmd_str, mea_tml, mea_act

def send_read_command(port, string, required_string=None, baudrate=115200, timeout=10):
    """
    Opens the serial port, sends a “hello” and then your command, and collects replies.
    Stops reading when either:
      • no more data arrives (an empty readline, i.e. per-read timeout fired), or
      • we see `required_string` in the response, if given, or
      • the overall `timeout` has elapsed.
    Returns the list of lines read so far.
    """
    lines = []
    start = time.monotonic()
    with serial.Serial(port, baudrate=baudrate, timeout=timeout) as ser:
        # hand-shake / mode set
        ser.setRTS(False)
        ser.flush()
        time.sleep(0.7)
        ser.write(b"hello\n")
        time.sleep(0.5)
        ser.write(f"{string}\n".encode())
        last_recv = time.monotonic()

        while True:
            # Check idle timeout
            now = time.monotonic()
            if now - last_recv >= timeout:
                print(f"No (additional) responses within user-set timeout: {timeout}s")
                return lines
            
            try:
                line = ser.readline().decode('utf-8', errors='replace').strip()
                # print(line)
                
                last_recv = time.monotonic() # Reset idle timer since we got data
            except Exception as e:
                print(f"Error reading line: {e!s}")
                break

            # if nothing arrived within per-read timeout, readline() returns b'' → line == ''
            if not line:
                print(f"No (additional) responses within user-set timeout: {timeout}s")
                return lines
                
            lines.append(line)

            # if we were waiting for a specific marker, stop when we see it
            if required_string and required_string in line:
                break
    return lines

def parse_command_blocks(lines, cmd_prefix="cmd:"):
    """
    Parse a list of lines and group them by commands that match `cmd_prefix`.

    The function identifies lines containing commands (e.g., 'cmd: something')
    and collects all subsequent lines into that command's "block" until the
    next command (or end of list).

    :param lines: List of strings to parse.
    :param cmd_prefix: The prefix that indicates a new command. Defaults to 'cmd:'.
    :return: A dictionary mapping { command_name: [list_of_lines_until_next_command] }
             If the same command appears multiple times, each occurrence will
             overwrite previous data unless modified to handle duplicates.
    """
    command_dict = {}
    current_cmd = None
    current_data = []

    for line in lines:
        # Try to find a line that includes 'cmd:' (possibly with extra characters).
        # The regex captures whatever follows 'cmd:' up to the first whitespace 
        # or end of string. E.g., 'cmd: hello' => group(1) = 'hello'.
        match = re.search(r'cmd:\s*([^\s]+)', line)

        if match:
            # If we have a "current_cmd", store its accumulated data first
            if current_cmd is not None:
                command_dict[current_cmd] = current_data

            # Extract the new command from the current line
            current_cmd = match.group(1)
            current_data = []  # Start a fresh list for data following this command
        else:
            # If this line does not declare a new command, and we've already seen a command,
            # then add the line to the current command's block
            if current_cmd is not None:
                current_data.append(line)

    # At the end, if there is a command in progress, save its data
    if current_cmd is not None:
        command_dict[current_cmd] = current_data

    return command_dict


def parse_serial(lines):
    """
    Efficiently parses a list of strings and returns a dictionary of lists for T, F, S, R, Sun, L.
    Skips lines that do not match the expected format.
    """
    import re
    pattern = re.compile(r"T:([\d\.]+),F:([\d\.]+),S:(\d+),R:(\d+),Sun:(\d+),L:(\d+)")
    result = {'T': [], 'F': [], 'S': [], 'R': [], 'Sun': [], 'L': []}
    for line in lines:
        m = pattern.search(line)
        if m:
            result['T'].append(float(m.group(1)))
            result['F'].append(float(m.group(2)))
            result['S'].append(int(m.group(3)))
            result['R'].append(int(m.group(4)))
            result['Sun'].append(int(m.group(5)))
            result['L'].append(int(m.group(6)))
    return result


def append_timeseries(base_series, *additional_series, additionalnumber=1):
    """
    Appends one or more additional time series to an existing base series,
    inserting a fixed gap (additionalnumber) between each appended series.

    :param base_series: List of time values (the initial series).
    :param additional_series: One or more lists of time values to append.
    :param additionalnumber: Gap to insert between each series (default 0).
    :return: A new merged list with all timeseries concatenated and offset, including gaps.
    """
    merged = list(base_series)
    offset = merged[-1] if merged else 0
    for s in additional_series:
        if len(s) == 0:
            continue
        # Always add the gap before appending a new series (except before the first)
        offset += additionalnumber
        adjusted = [t + offset for t in s]
        merged.extend(adjusted)
        offset = adjusted[-1]
    return merged




In [ ]:
PORT = findDevice(question="hello", answer="Ready")  #<----automatic find ambit
# PORT = "COM5"  #<--- manual port selection (overrides the auto-detect above)

In [ ]:
SAVE = False
mytime = datetime.now().strftime("%y%m%d_%H%M_%S")

cmdA = []
cmdA += gen_cmd_arr_line(50, 10, 0)  # 50 points, 10Hz, no actinic light
cmdA += gen_cmd_arr_line(8, 10, 255)  # 8 points, 10Hz, no actinic light
cmdA += gen_cmd_arr_line(50, 10, 0)  # 50 points, 10Hz, no actinic light

cmd_strA, timelineA, act_arrA = calc_arr_param(cmdA, 0)
act_arrA = calc_arr_param(cmdA, 0)
data_serial = send_read_command(PORT, cmd_strA, required_string="Done")
parsed_serial = parse_serial(data_serial)

df = pd.DataFrame(parsed_serial)
df.insert(0,"time (sec)",timelineA)


plt.plot(df["time (sec)"],df["F"])
plt.xlabel("Time (sec)")
plt.ylabel("Fluo. rel. yield")
plt.title("Fluorescence Intensity over pulses")


if SAVE:
    df.to_csv(f"{mytime}_output.csv",sep=",", index=False)


In [ ]:
cmdA = []
cmdA += gen_cmd_arr_line(500, 500, 255)  # 400 points, 200Hz, 255 actinic light (on scale of 0-255)

cmd_strA, timelineA, act_arrA = calc_arr_param(cmdA, 0)
data_serial = send_read_command(PORT, cmd_strA, required_string="Done")
parsed_serial = parse_serial(data_serial)

df = pd.DataFrame(parsed_serial)
df.insert(0,"time (sec)",timelineA)


plt.plot(df["time (sec)"],df["F"])
plt.xscale("log")
plt.xlabel("Time (sec)")
plt.ylabel("Intensity")
plt.title("Fluorescence Intensity over pulses")


In [ ]:

def STING(PORT, POINT_LOW = 10, FREQ_LOW = 10, POINT_HIGH = 40):
    cmdA = gen_cmd_arr_line(POINT_LOW, FREQ_LOW, 0)
    cmdA += gen_cmd_arr_line(POINT_HIGH, 1000, 0)
    cmdA += gen_cmd_arr_line(POINT_LOW, FREQ_LOW, 0)
    cmdA += gen_cmd_arr_line(POINT_HIGH, 1000, 0)
    cmdA += gen_cmd_arr_line(POINT_LOW, FREQ_LOW, 0)
    cmdA += gen_cmd_arr_line(POINT_HIGH, 1000, 0)
    cmdA += gen_cmd_arr_line(POINT_LOW, FREQ_LOW, 0)
    cmdA += gen_cmd_arr_line(POINT_HIGH, 1000, 0)
    cmdA += gen_cmd_arr_line(POINT_LOW, FREQ_LOW, 0)
    cmd_strA, timelineA, _ = calc_arr_param(cmdA, 0)
    data_serial = send_read_command(PORT, cmd_strA, required_string="Done")
    parsed_serial = parse_serial(data_serial)
    fluo =np.array(parsed_serial["F"])
    sun = np.array(parsed_serial["S"])
    leaf = np.array(parsed_serial["L"])
    return fluo, sun, leaf


results = []
for freq_low in [10,20,40,80,160]:
    # time.sleep(30)
    fluo, sun, leaf = STING(PORT, FREQ_LOW=freq_low)
    results.append((freq_low,fluo, sun, leaf))
    plt.plot(fluo,label=f"Frequency low: {freq_low} Hz")
    plt.legend()



In [ ]:


results = []
for freq_low in [10,20,40,80,160]:
    time.sleep(10)
    fluo, sun, leaf, timeline = STING(PORT, FREQ_LOW=freq_low)
    results.append((freq_low,fluo, sun, leaf, timeline))
    plt.plot(fluo, label=f"Frequency low: {freq_low} Hz")
    plt.legend()



In [ ]:


results = []
for freq_low in [10,20,40,80,160]:
    time.sleep(10)
    fluo, sun, leaf, timeline = STING(PORT, FREQ_LOW=freq_low)
    results.append((freq_low,fluo, sun, leaf, timeline))
    plt.plot(timeline, fluo, label=f"Frequency low: {freq_low} Hz")
    plt.legend()



In [ ]:
cmdA = gen_cmd_arr_line(1, 120, 0)
# cmdA += gen_cmd_arr_line(10, 100, 0)
cmd_strA, timelineA, _ = calc_arr_param(cmdA, 0)
data_serial = send_read_command(PORT, cmd_strA, required_string="Done")
data_serial

In [ ]:

def STING(PORT, POINT_LOW = 10, FREQ_LOW = 10, POINT_HIGH = 40):
    cmdA = gen_cmd_arr_line(POINT_LOW, FREQ_LOW, 0)
    cmdA += gen_cmd_arr_line(POINT_HIGH, 1000, 0)
    cmdA += gen_cmd_arr_line(POINT_LOW, FREQ_LOW, 0)
    cmdA += gen_cmd_arr_line(POINT_HIGH, 1000, 0)
    cmdA += gen_cmd_arr_line(POINT_LOW, FREQ_LOW, 0)
    cmdA += gen_cmd_arr_line(POINT_HIGH, 1000, 0)
    cmdA += gen_cmd_arr_line(POINT_LOW, FREQ_LOW, 0)
    cmdA += gen_cmd_arr_line(POINT_HIGH, 1000, 0)
    cmdA += gen_cmd_arr_line(POINT_LOW, FREQ_LOW, 0)
    cmd_strA, timelineA, _ = calc_arr_param(cmdA, 0)
    data_serial = send_read_command(PORT, cmd_strA, required_string="Done")
    parsed_serial = parse_serial(data_serial)
    fluo =np.array(parsed_serial["F"])
    sun = np.array(parsed_serial["S"])
    leaf = np.array(parsed_serial["L"])
    return fluo, sun, leaf

results = []
for freq_low in [10,20,40,80,160]:
    time.sleep(10)
    fluo, sun, leaf, timeline = STING(PORT, FREQ_LOW=freq_low)
    results.append((freq_low,fluo, sun, leaf, timeline))
    plt.plot(timeline, fluo, label=f"Frequency low: {freq_low} Hz")
    plt.legend()

